# Preprocesamiento de los datos

Este notebook lee el resultado del EDA, aplica las transformaciones previas al entrenamiento y guarda los conjuntos de entrenamiento y prueba. La clase se mantiene dentro de ambos archivos.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path


def localizar_raiz_proyecto():
    """Localiza la carpeta principal tanto desde VS Code como desde Jupyter."""
    carpeta_actual = Path.cwd().resolve()

    if (carpeta_actual / "datos").exists():
        return carpeta_actual

    if (carpeta_actual.parent / "datos").exists():
        return carpeta_actual.parent

    raise FileNotFoundError(
        "No se encuentra la carpeta 'datos'. Abre el notebook desde el repositorio."
    )


RAIZ_PROYECTO = localizar_raiz_proyecto()


In [2]:
RUTA_ENTRADA = RAIZ_PROYECTO / "datos" / "intermedios" / "dataset_eda.csv"
RUTA_TRAIN = RAIZ_PROYECTO / "datos" / "procesados" / "train.csv"
RUTA_TEST = RAIZ_PROYECTO / "datos" / "procesados" / "test.csv"

RUTA_TRAIN.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RUTA_ENTRADA, dtype={"id": "string"})

print("Conjunto leído desde:", RUTA_ENTRADA)
print("Dimensiones:", df.shape)


Conjunto leído desde: C:\Users\frana\Desktop\codigo\prediccion-enfermedades-tiroideas\datos\intermedios\dataset_eda.csv
Dimensiones: (9172, 32)


### Eliminación de variables no utilizadas

In [3]:
df_preprocesado = df.copy()

variables_numericas = [
    "edad", "TSH", "T3", "TT4", "T4U", "FTI", "TBG"
]

# Convertimos las variables numéricas
df_preprocesado[variables_numericas] = (
    df_preprocesado[variables_numericas]
    .apply(pd.to_numeric, errors="coerce")
)

columnas_eliminar = [
    "id",
    "diagnostico",
    "fuente",
    "TBG",
    "TBG_medida"
]

df_preprocesado = df_preprocesado.drop(
    columns=columnas_eliminar
)

print(df_preprocesado.shape)
display(df_preprocesado.head())

(9172, 27)


,edad,sexo,tratamiento_con_tiroxina,consulta_sobre_tiroxina,medicacion_antitiroidea,enfermo,embarazada,cirugia_tiroidea,tratamiento_I131,consulta_hipotiroidismo,...,TSH,T3_medida,T3,TT4_medida,TT4,T4U_medida,T4U,FTI_medido,FTI,clase
0,29,F,f,f,f,f,f,f,f,t,...,0.3,f,NaN,f,NaN,f,NaN,f,NaN,otro
1,29,F,f,f,f,f,f,f,f,f,...,1.6,t,1.9,t,128.0,f,NaN,f,NaN,otro
2,41,F,f,f,f,f,f,f,f,f,...,NaN,f,NaN,f,NaN,f,NaN,f,NaN,otro
3,36,F,f,f,f,f,f,f,f,f,...,NaN,f,NaN,f,NaN,f,NaN,f,NaN,otro
4,32,F,f,f,f,f,f,f,f,f,...,NaN,f,NaN,f,NaN,f,NaN,f,NaN,otro


### Tratamiento de valores incorrectos

In [4]:
# Las edades superiores a 110 se consideran incorrectas
condicion_edad = (
    (df_preprocesado["edad"] < 0) |
    (df_preprocesado["edad"] > 110)
)

print("Edades incorrectas:", condicion_edad.sum())

df_preprocesado.loc[
    condicion_edad,
    "edad"
] = np.nan

Edades incorrectas: 4


### Codificación de las variables binarias

In [5]:
variables_binarias = [
    "tratamiento_con_tiroxina",
    "consulta_sobre_tiroxina",
    "medicacion_antitiroidea",
    "enfermo",
    "embarazada",
    "cirugia_tiroidea",
    "tratamiento_I131",
    "consulta_hipotiroidismo",
    "consulta_hipertiroidismo",
    "litio",
    "bocio",
    "tumor",
    "hipopituitarismo",
    "trastorno_psiquiatrico",
    "TSH_medida",
    "T3_medida",
    "TT4_medida",
    "T4U_medida",
    "FTI_medido"
]

# t se representa con 1 y f con 0
for variable in variables_binarias:
    df_preprocesado[variable] = df_preprocesado[variable].map({
        "t": 1,
        "f": 0
    })

# Codificación del sexo
df_preprocesado["sexo"] = df_preprocesado["sexo"].map({
    "F": 0,
    "M": 1
})

display(df_preprocesado)

,edad,sexo,tratamiento_con_tiroxina,consulta_sobre_tiroxina,medicacion_antitiroidea,enfermo,embarazada,cirugia_tiroidea,tratamiento_I131,consulta_hipotiroidismo,...,TSH,T3_medida,T3,TT4_medida,TT4,T4U_medida,T4U,FTI_medido,FTI,clase
0,29.0,0.0,0,0,0,0,0,0,0,1,...,0.3,0,NaN,0,NaN,0,NaN,0,NaN,otro
1,29.0,0.0,0,0,0,0,0,0,0,0,...,1.6,1,1.9,1,128.0,0,NaN,0,NaN,otro
2,41.0,0.0,0,0,0,0,0,0,0,0,...,NaN,0,NaN,0,NaN,0,NaN,0,NaN,otro
3,36.0,0.0,0,0,0,0,0,0,0,0,...,NaN,0,NaN,0,NaN,0,NaN,0,NaN,otro
4,32.0,0.0,0,0,0,0,0,0,0,0,...,NaN,0,NaN,0,NaN,0,NaN,0,NaN,otro
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9167,56.0,1.0,0,0,0,0,0,0,0,0,...,NaN,0,NaN,1,64.0,1,0.83,1,77.0,otro
9168,22.0,1.0,0,0,0,0,0,0,0,0,...,NaN,0,NaN,1,91.0,1,0.92,1,99.0,otro
9169,69.0,1.0,0,0,0,0,0,0,0,0,...,NaN,0,NaN,1,113.0,1,1.27,1,89.0,otro
9170,47.0,0.0,0,0,0,0,0,0,0,0,...,NaN,0,NaN,1,75.0,1,0.85,1,88.0,otro


In [6]:
print(
    df_preprocesado[
        variables_binarias + ["sexo"]
    ].isna().sum()
)

tratamiento_con_tiroxina      0
consulta_sobre_tiroxina       0
medicacion_antitiroidea       0
enfermo                       0
embarazada                    0
cirugia_tiroidea              0
tratamiento_I131              0
consulta_hipotiroidismo       0
consulta_hipertiroidismo      0
litio                         0
bocio                         0
tumor                         0
hipopituitarismo              0
trastorno_psiquiatrico        0
TSH_medida                    0
T3_medida                     0
TT4_medida                    0
T4U_medida                    0
FTI_medido                    0
sexo                        307
dtype: int64


## División de entrenamiento y prueba

In [7]:
from sklearn.model_selection import train_test_split

# La división se hace manteniendo la proporción de las tres clases
df_train, df_test = train_test_split(
    df_preprocesado,
    test_size=0.20,
    random_state=42,
    stratify=df_preprocesado["clase"]
)

print("Registros de entrenamiento:", df_train.shape[0])
print("Registros de prueba:", df_test.shape[0])


Registros de entrenamiento: 7337
Registros de prueba: 1835


In [8]:
print("Distribución en entrenamiento:")
display(df_train["clase"].value_counts(normalize=True).round(3))

print("Distribución en prueba:")
display(df_test["clase"].value_counts(normalize=True).round(3))


Distribución en entrenamiento:


clase
otro               0.903
hipotiroidismo     0.072
hipertiroidismo    0.025
Name: proportion, dtype: float64

Distribución en prueba:


clase
otro               0.903
hipotiroidismo     0.072
hipertiroidismo    0.025
Name: proportion, dtype: float64

## Guardado de los conjuntos

In [9]:
# Se guardan las variables y la clase en el mismo archivo
df_train.to_csv(RUTA_TRAIN, index=False)
df_test.to_csv(RUTA_TEST, index=False)

print("Entrenamiento guardado en:", RUTA_TRAIN)
print("Prueba guardada en:", RUTA_TEST)


Entrenamiento guardado en: C:\Users\frana\Desktop\codigo\prediccion-enfermedades-tiroideas\datos\procesados\train.csv
Prueba guardada en: C:\Users\frana\Desktop\codigo\prediccion-enfermedades-tiroideas\datos\procesados\test.csv


**Importante:** los valores ausentes no se imputan ni las variables se escalan antes de guardar estos archivos. Estos pasos se incluirán dentro de los pipelines de los modelos, aprendiendo sus parámetros únicamente con los datos de entrenamiento.